# Первоначальная обработка данных

Проект «Подбор домов Красноярска по анкете». Одна запись — оценка адреса по анкете. Целевой ответ — «Добавили бы вы этот дом в список для просмотра?».

Каталог содержит реальные адреса и расстояния по прямой. Анкеты и ответы сгенерированы для учебного исследования. Запускайте все ячейки. CSV берутся из папки **data** рядом с ноутбуком.

[Данные каталога © OpenStreetMap contributors, ODbL](https://www.openstreetmap.org/copyright).

## Загрузка и проверка

Каталог собран из листа «Данные» книги **Krasnoyarsk_baza_zhilykh_domov_orig (1).xlsx**. Согласно листу «Методика», снимок сформирован 18.11.2026 из OpenStreetMap. Генератор **prepare_data.py** воспроизводит три CSV с seed=42.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

project = Path.cwd()
if (project / 'house_matching' / 'prepare_data.py').is_file():
    project = project / 'house_matching'
sys.path.insert(0, str(project))

from prepare_data import (
    DISTANCES, FEATURES, HOUSES_PER_SURVEY, LIMITS, PARKING, PLACES,
    PRIORITIES, SEED, SURVEY_COUNT, TARGET,
)

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
blue, teal, terracotta, ink, grid = '#2964A3', '#087E77', '#B85B34', '#162B41', '#DCE6EF'
places = [russian for _, russian in PLACES.values()]
short = dict(zip(DISTANCES + LIMITS + PRIORITIES,
                 [f'{name} дома' for name in places] + [f'{name} в анкете' for name in places]
                 + [f'{name}: важность' for name in places + ['Парковка']]))
short['parking'] = 'Парковка'

houses = pd.read_csv(project / 'data' / 'houses.csv', dtype={'floors': 'Int64'})
surveys = pd.read_csv(project / 'data' / 'surveys.csv')
answers = pd.read_csv(project / 'data' / 'answers.csv')
pairs = answers.merge(houses, on='house_id', validate='many_to_one')
pairs = pairs.merge(surveys, on='survey_id', validate='many_to_one')

assert len(FEATURES) == 23 and len(surveys) == SURVEY_COUNT
assert len(pairs) == SURVEY_COUNT * HOUSES_PER_SURVEY
assert not pairs.duplicated(['survey_id', 'house_id']).any()
assert pairs.groupby('survey_id').size().eq(HOUSES_PER_SURVEY).all()
assert pairs[TARGET].isin([0, 1]).all()
fits_district = [district in district_list.split('|')
                 for district, district_list in zip(pairs['district'], pairs['districts'])]
assert all(fits_district)

train_ids, test_ids = train_test_split(surveys['survey_id'], test_size=.2, random_state=SEED)
assert set(train_ids).isdisjoint(test_ids)
eda = pairs.loc[pairs['survey_id'].isin(train_ids)].copy()
print(f'{len(houses):,} адресов · {len(surveys)} анкет · {len(pairs):,} ответов'.replace(',', ' '))
print(f'Для анализа отклика {len(train_ids)} анкет, для финального теста {len(test_ids)}.')
eda[FEATURES + [TARGET]].info()
display(eda[['survey_id', 'address', 'stop_m', 'want_stop_m', 'priority_stop', TARGET]].head())

Проверка загрузки подтверждает, что расстояния и оценки важности читаются как числа.
Категория парковки хранится текстом. Одинаковый адрес может встречаться у нескольких анкет,
а уникальность строки задаёт пара survey_id и house_id. Отложенные анкеты используются в финальном тесте.

In [ ]:
print(f'Неизвестная этажность в карточках: {houses["floors"].isnull().sum():,}.'.replace(',', ' '))
print('Повторы строки с той же анкетой и домом:', pairs.duplicated(['survey_id', 'house_id']).sum())
print('Совпавшие наборы из 23 признаков:', eda[FEATURES].duplicated().sum())
display(eda[FEATURES + [TARGET]].describe(include='all').T.round(2).rename(index=short))
display(houses['parking'].value_counts().rename('Адресов').to_frame())
display(eda[TARGET].value_counts().sort_index().rename('Ответов').to_frame().rename(index={0: 'Нет', 1: 'Да'}))

## Распределения, корреляции, качество данных и связь с откликом

### Распределения · параметры домов

Семь расстояний по прямой и категория парковки. В каждом графике единица наблюдения — адрес.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12.6, 5.2), layout='constrained')
for ax, feature, title in zip(axes.flat, DISTANCES, places):
    houses[feature].hist(ax=ax, bins=25, color=blue, grid=False)
    ax.axvline(houses[feature].median(), color=terracotta, lw=1.5)
    ax.set(title=title, xlabel='м', ylabel='Адресов')
park_count = houses['parking'].value_counts().reindex(PARKING)
axes.flat[7].bar(['≤ 150', '150–400', '> 400'], park_count, color=teal)
axes.flat[7].set(title='Парковка', xlabel='м', ylabel='Адресов')
plt.show()

medians = houses[DISTANCES].median()
display(Markdown(
    f'Медиана расстояния до остановки **{medians["stop_m"]:.0f} м**, '
    f'до ТЦ **{medians["mall_m"]:.0f} м**. Ближайшая парковка в пределах 150 м '
    f'есть на карте для **{park_count.iloc[0] / len(houses):.1%}** адресов.'
))

### Распределения · желаемые расстояния

Для каждой характеристики анкеты заданы три варианта, полученные из квартилей каталога.
Единица наблюдения — анкета.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12.6, 5.1), layout='constrained')
for ax, feature, title in zip(axes.flat, LIMITS, places):
    counts = surveys[feature].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts, color=teal)
    ax.set(title=title, xlabel='Желаемое, м', ylabel='Анкет')
axes.flat[7].set_visible(False)
plt.show()

display(Markdown(
    f'Медианный порог в учебных анкетах составляет **{surveys["want_stop_m"].median():.0f} м** '
    f'для остановки и **{surveys["want_mall_m"].median():.0f} м** для ТЦ.'
))

### Распределения · важность критериев

0 означает отсутствие пожелания. 3 означает высокий приоритет.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12.6, 5.1), layout='constrained')
for ax, feature, title in zip(axes.flat, PRIORITIES, places + ['Парковка']):
    counts = surveys[feature].value_counts().reindex(range(4), fill_value=0)
    ax.bar(counts.index, counts, color=blue)
    ax.set(title=title, xlabel='Важность', ylabel='Анкет', xticks=range(4))
plt.show()

high = surveys[PRIORITIES].ge(2).mean()
display(Markdown(
    f'Приоритет 2 или 3 указан у **{high["priority_school"]:.1%}** анкет для школы '
    f'и у **{high["priority_parking"]:.1%}** для парковки.'
))

### Краткие выводы по 23 распределениям

In [ ]:
distribution_notes = {}
for field in DISTANCES:
    values = houses[field]
    distribution_notes[field] = (
        f'Медиана {values.median():.0f} м, у 90% адресов до {values.quantile(.9):.0f} м, '
        f'максимум {values.max():.0f} м.'
    )
distribution_notes['parking'] = (
    f'Чаще всего «{park_count.idxmax()}», доля {park_count.max() / len(houses):.1%}.'
)
for field in LIMITS:
    counts = surveys[field].value_counts()
    distribution_notes[field] = (
        f'Уровни {counts.size}, самый частый порог {counts.idxmax()} м '
        f'({counts.max() / len(surveys):.1%} анкет).'
    )
for field in PRIORITIES:
    values = surveys[field]
    distribution_notes[field] = (
        f'Уровень {values.mode().iloc[0]} встречается чаще всего. Важность 2–3 у {values.ge(2).mean():.1%} анкет.'
    )
display(pd.Series(distribution_notes, name='Наблюдение').rename(index=short).to_frame())

### Корреляции · матрица Спирмена

Матрица рассчитана по числовым признакам пар «анкета × дом» в аналитической выборке.
Толстые линии отмечают группы расстояний, желаемых расстояний и оценок важности.

In [ ]:
numeric = DISTANCES + LIMITS + PRIORITIES
rho = eda[numeric].corr(method='spearman')
matrix_labels = places + [f'{name} (порог)' for name in places] + [
    f'{name} (вес)' for name in places + ['Парковка']
]
fig, ax = plt.subplots(figsize=(9, 6.1), layout='constrained')
image = ax.imshow(rho, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(numeric)), matrix_labels, rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(len(numeric)), matrix_labels, fontsize=8)
for boundary in (6.5, 13.5):
    ax.axhline(boundary, lw=1.4, color=ink)
    ax.axvline(boundary, lw=1.4, color=ink)
fig.colorbar(image, ax=ax, shrink=.85, label='Коэффициент Спирмена')
plt.show()

### Корреляции · сильные связи расстояний

Две пары с наибольшим абсолютным коэффициентом в каталоге. Цвет показывает плотность адресов.

In [ ]:
catalog_rho = houses[DISTANCES].corr(method='spearman')
upper = np.triu(np.ones(catalog_rho.shape, dtype=bool), k=1)
relations = catalog_rho.where(upper).stack()
strongest = relations.reindex(relations.abs().nlargest(2).index)
names = dict(zip(DISTANCES, places))
fig, axes = plt.subplots(1, 2, figsize=(11.7, 4.3), layout='constrained')
for ax, ((x, y), value) in zip(axes, strongest.items()):
    image = ax.hexbin(houses[x], houses[y], gridsize=30, mincnt=1, bins='log', cmap='Blues')
    ax.set(xlabel=f'{names[x]}, м', ylabel=f'{names[y]}, м', title=f'ρ = {value:.2f}')
    fig.colorbar(image, ax=ax, label='Адресов')
plt.show()

display(Markdown(
    f'У школы и детсада **ρ = {catalog_rho.loc["school_m", "kindergarten_m"]:.2f}**. '
    'У многих адресов близость одного объекта сопровождается близостью второго.'
))

### Статистика · расстояния до объектов

Ящик показывает средние 50% значений, линия внутри него — медиану.
Точки за усами отмечены правилом 1,5 IQR.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12.6, 4.3), layout='constrained')
for ax, feature, title in zip(axes.flat, DISTANCES, places):
    houses.boxplot(column=feature, ax=ax, tick_labels=[''], vert=False,
                   flierprops={'markersize': 2, 'markeredgecolor': terracotta, 'alpha': .35})
    ax.set(title=title, xlabel='м')
    ax.set_yticks([])
axes.flat[7].set_visible(False)
plt.show()

### Статистика · параметры анкет

Boxplot обобщает ответы. Дискретные уровни важности и разрешённые пороги проверяются по их шкале.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.2), layout='constrained')
surveys[LIMITS].rename(columns=dict(zip(LIMITS, places))).boxplot(ax=axes[0], vert=False)
surveys[PRIORITIES].rename(columns=dict(zip(PRIORITIES, places + ['Парковка']))).boxplot(
    ax=axes[1], vert=False
)
axes[0].set(title='Желаемые расстояния', xlabel='м')
axes[1].set(title='Важность', xlabel='Балл', xticks=range(4))
plt.show()

### Статистика · качество всех полей

Для признаков дома количество значений считается по каталогу, для параметров анкеты — по анкетам.
Допустимые дальние расстояния описывают адреса с низкой доступностью.

In [ ]:
def quality_table(frame, numeric_fields, invalid):
    table = pd.DataFrame({
        'Пропусков': frame.isnull().sum(),
        'Различных': frame.nunique(),
        'Одиночных': [frame[col].value_counts().eq(1).sum() for col in frame],
        'Вне шкалы': invalid.sum().reindex(frame.columns, fill_value=0),
        'За усами': 0,
    })
    values = frame[numeric_fields]
    q1, q3 = values.quantile([.25, .75]).to_numpy()
    iqr = q3 - q1
    table.loc[numeric_fields, 'За усами'] = (
        values.lt(q1 - 1.5 * iqr) | values.gt(q3 + 1.5 * iqr)
    ).sum()
    return table


bad_house = houses[DISTANCES].lt(0) | ~np.isfinite(houses[DISTANCES])
bad_house['parking'] = ~houses['parking'].isin(PARKING)
bad_survey = surveys[LIMITS].le(0).join(~surveys[PRIORITIES].isin([0, 1, 2, 3]))
quality = pd.concat([
    quality_table(houses[DISTANCES + ['parking']], DISTANCES, bad_house),
    quality_table(surveys[LIMITS + PRIORITIES], LIMITS + PRIORITIES, bad_survey),
])
quality.loc[PRIORITIES, 'За усами'] = 0
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout='constrained')
quality.loc[DISTANCES, ['За усами', 'Одиночных']].rename(index=short).plot.barh(
    ax=axes[0], color=[terracotta, blue], xlabel='Адресов', title='Редкие и крайние расстояния')
quality['Различных'].loc[LIMITS + PRIORITIES + ['parking']].rename(index=short).plot.barh(
    ax=axes[1], color=teal, xlabel='Значений', title='Уровни анкет и парковки')
for ax in axes:
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=.25)
plt.show()
display(Markdown(
    f'В 23 признаках **{quality["Пропусков"].sum()} пропусков** и '
    f'**{quality["Вне шкалы"].sum()} значений вне допустимой шкалы**. '
    f'У поля «Остановка» {quality.loc["stop_m", "За усами"]} адресов за усами.'
))

In [ ]:
display(quality.rename(index=short))
display(houses[DISTANCES].eq(0).sum().rename('Нулевых расстояний').to_frame().rename(index=short))

### Кодирование категорий и повторная проверка

Для парковки создаём отдельный бинарный столбец на категорию. Числа пока остаются в исходных шкалах.

In [ ]:
encoded = pd.get_dummies(eda[FEATURES], columns=['parking'], dtype='int8')
indicator_fields = encoded.columns.difference(numeric).tolist()
assert encoded[indicator_fields].sum(axis=1).eq(1).all()
encoded.info()
display(encoded.head(3))
display(encoded.describe().T.round(2))

invalid_encoded = encoded[DISTANCES].lt(0).join(encoded[LIMITS].le(0))
invalid_encoded = invalid_encoded.join(~encoded[PRIORITIES + indicator_fields].isin([0, 1, 2, 3]))
invalid_encoded[indicator_fields] = ~encoded[indicator_fields].isin([0, 1])
encoded_quality = quality_table(encoded, DISTANCES + LIMITS, invalid_encoded)
display(encoded_quality.rename(index=short))
print('Совпавшие строки признаков после кодирования:', encoded.duplicated().sum())

fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.2), layout='constrained')
eda['parking'].value_counts().reindex(PARKING).plot.barh(ax=axes[0], color=blue, title='Категории до')
encoded[indicator_fields].sum().rename(lambda name: name.removeprefix('parking_')).reindex(PARKING).plot.barh(
    ax=axes[1], color=teal, title='Число единиц после'
)
for ax in axes:
    ax.set_xlabel('Пар «анкета × дом»')
plt.show()

Сумма единиц в новых столбцах воспроизводит частоты категорий. Для бинарных столбцов проверяется диапазон 0–1.
Признаки времени в проекте отсутствуют, а расстояния проверяются по неотрицательности.

### Наблюдение 1 · выполнение порога и ответ анкеты

Для каждого объекта смотрим пары, где его важность равна 2 или 3. Сравниваем долю учебных ответов «да»
при соблюдении и превышении желаемого расстояния.

In [ ]:
def summarize_answers(frame, columns):
    summary = frame.groupby(columns, observed=True)[TARGET].agg(n='size', yes='sum')
    return summary.assign(rate=100 * summary['yes'] / summary['n'])


rows = []
for distance, limit, priority, name in zip(DISTANCES, LIMITS, PRIORITIES, places):
    group = eda.loc[eda[priority].ge(2)].copy()
    group['within_limit'] = group[distance].le(group[limit])
    summary = summarize_answers(group, 'within_limit').reindex([False, True])
    rows.append({
        'Объект': name, 'Дальше, n': summary.loc[False, 'n'], 'В пределах, n': summary.loc[True, 'n'],
        'Дальше, %': summary.loc[False, 'rate'], 'В пределах, %': summary.loc[True, 'rate'],
    })
threshold_stats = pd.DataFrame(rows).set_index('Объект')
threshold_stats['Разница, п. п.'] = threshold_stats['В пределах, %'] - threshold_stats['Дальше, %']

ax = threshold_stats[['Дальше, %', 'В пределах, %']].plot.barh(
    figsize=(9.4, 4.2), color=[terracotta, teal], xlim=(0, 100), xlabel='Ответов «да», %',
)
ax.invert_yaxis()
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.show()

strong = threshold_stats['Разница, п. п.'].idxmax()
row = threshold_stats.loc[strong]
display(Markdown(
    f'Наибольшая разница в этом сравнении у критерия **«{strong}»**. При соблюдении порога '
    f'**{row["В пределах, %"]:.1f}%** ответов «да» (n={row["В пределах, n"]:.0f}) '
    f'против **{row["Дальше, %"]:.1f}%** (n={row["Дальше, n"]:.0f}). '
    f'Разница **{row["Разница, п. п."]:.1f} п. п.**'
))

In [ ]:
display(threshold_stats.round(1))

### Наблюдение 2 · сочетания условий

Сначала сравниваем школу и детсад среди анкет, где важны оба объекта.
Затем сравниваем категории парковки отдельно при низком и высоком приоритете.

In [ ]:
education = eda.loc[eda['priority_school'].ge(2) & eda['priority_kindergarten'].ge(2)].copy()
education['school_near'] = education['school_m'].le(education['want_school_m'])
education['kindergarten_near'] = education['kindergarten_m'].le(education['want_kindergarten_m'])
education_order = [(False, False), (True, False), (False, True), (True, True)]
education_stats = summarize_answers(education, ['school_near', 'kindergarten_near']).reindex(education_order)
education_stats.index = ['Оба дальше', 'Только школа', 'Только детсад', 'Оба рядом']

parking_groups = eda.assign(parking_priority=np.where(eda['priority_parking'].ge(2), '2–3', '0–1'))
parking_stats = summarize_answers(parking_groups, ['parking_priority', 'parking'])
parking_rates = parking_stats['rate'].unstack(0).reindex(PARKING)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.35), layout='constrained')
bars = axes[0].bar(education_stats.index, education_stats['rate'], color=[terracotta, blue, blue, teal])
axes[0].bar_label(bars, labels=[f'{rate:.0f}%\nn={n}' for rate, n in
                              zip(education_stats['rate'], education_stats['n'])], padding=4, fontsize=9)
axes[0].tick_params(axis='x', labelrotation=15)
axes[0].set(title='Школа и детсад', ylim=(0, 100), ylabel='Ответов «да», %')
parking_rates.rename(index=dict(zip(PARKING, ['≤ 150 м', '150–400 м', '> 400 м']))).plot.bar(
    ax=axes[1], color=[blue, teal], rot=0, xlabel='Расстояние', ylabel='Ответов «да», %',
    title='Парковка при разных приоритетах', ylim=(0, 100),
)
axes[1].legend(title='Важность парковки')
plt.show()

education_gap = education_stats.loc['Оба рядом', 'rate'] - education_stats.loc['Оба дальше', 'rate']
parking_gaps = parking_rates.iloc[0] - parking_rates.iloc[-1]
display(Markdown(
    f'Школа и детсад рядом одновременно дают разницу **{education_gap:.1f} п. п.** '
    f'относительно группы «оба дальше». Разница между ближайшей и дальней парковкой '
    f'составляет **{parking_gaps["2–3"]:.1f} п. п.** при важности 2–3 '
    f'и **{parking_gaps["0–1"]:.1f} п. п.** при важности 0–1.'
))

In [ ]:
display(education_stats.round(1))
display(parking_stats.round(1))

Числа в группах относятся к парам «анкета × дом». Одна анкета оценивает несколько адресов.
Эти доли описывают ответы учебных анкет. На совместную близость школы и детсада влияет и расположение
инфраструктуры, и правило генерации учебного выбора.